# AutoML Notebook Training

In [1]:
import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import accuracy_score, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC, SVR

In [2]:
class AutoMLPipeline:
    def __init__(self, df, target, model_path="/Users/basudev/Documents/Auto ML/automl-project/models/best_model.pkl"):
        self.df = df
        self.target = target
        self.model_path = model_path
        self.task_type = None
        self.best_model = None
        self.best_score = -np.inf
        self.results = []

    def detect_task(self):
        y = self.df[self.target]
        if y.dtype == "object" or y.nunique() < 20:
            self.task_type = "classification"
        else:
            self.task_type = "regression"
        print(f"Detected Task: {self.task_type}")

    def split_data(self):
        X = self.df.drop(columns=[self.target])
        y = self.df[self.target]
        return train_test_split(X, y, test_size=0.2, random_state=42)

    def build_preprocessor(self, X):
        numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
        categorical_features = X.select_dtypes(include=["object"]).columns

        numeric_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler())
        ])

        categorical_pipeline = Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ])

        return ColumnTransformer([
            ("num", numeric_pipeline, numeric_features),
            ("cat", categorical_pipeline, categorical_features)
        ])

    def get_models(self):
        if self.task_type == "classification":
            return {
                "LogisticRegression": LogisticRegression(max_iter=1000),
                "SVM": SVC(probability=True)
            }
        return {
            "Ridge": Ridge(),
            "SVR": SVR()
        }

    def evaluate(self, model, X_test, y_test):
        preds = model.predict(X_test)
        if self.task_type == "classification":
            return accuracy_score(y_test, preds)
        return r2_score(y_test, preds)

    def train(self):
        self.detect_task()
        X_train, X_test, y_train, y_test = self.split_data()
        preprocessor = self.build_preprocessor(X_train)
        models = self.get_models()

        for name, model in models.items():
            pipeline = Pipeline([
                ("preprocessor", preprocessor),
                ("model", model)
            ])

            scores = cross_val_score(pipeline, X_train, y_train, cv=5)
            pipeline.fit(X_train, y_train)
            test_score = self.evaluate(pipeline, X_test, y_test)
            avg_score = np.mean(scores)

            self.results.append({
                "model": name,
                "cv_score": round(avg_score, 4),
                "test_score": round(test_score, 4),
                "best": "This is the best fit model" if avg_score > self.best_score else ""
            })

            print(f"{name} | CV Score: {avg_score:.4f} | Test Score: {test_score:.4f}")

            if avg_score > self.best_score:
                self.best_score = avg_score
                self.best_model = pipeline

        print("\nBest Model Selected!")

    def save_model(self):
        os.makedirs(os.path.dirname(self.model_path), exist_ok=True)
        joblib.dump(self.best_model, self.model_path)
        print(f"Model saved at {self.model_path}")

    def run(self):
        self.train()
        self.save_model()
        return pd.DataFrame(self.results)

In [3]:
df = pd.read_csv("/Users/basudev/Downloads/data.csv")
TARGET_COLUMN = "price"

automl = AutoMLPipeline(df, target=TARGET_COLUMN)
results = automl.run()

print("\nModel Comparison:")
print(results)

Detected Task: regression


Ridge | CV Score: 0.6914 | Test Score: 0.0539


SVR | CV Score: -0.0472 | Test Score: -0.0134

Best Model Selected!
Model saved at /Users/basudev/Documents/Auto ML/automl-project/models/best_model.pkl

Model Comparison:
   model  cv_score  test_score                        best
0  Ridge    0.6914      0.0539  This is the best fit model
1    SVR   -0.0472     -0.0134                            
